# ETF Lookthrough Example

This notebook previews the cached US sector lookthrough for `QQQ`, then optionally refreshes it from Alpha Vantage.

Default behavior is practical for local debugging: it targets `QQQ`, attempts a live fetch when an `ALPHA_VANTAGE_API_KEY` is available, and falls back cleanly to the cached JSON store if the network is unavailable.

In [1]:
from pathlib import Path
import os
import sys
from typing import Dict, Iterable

project_root = Path.cwd().resolve()
while not (project_root / "market_helper").exists():
    if project_root.parent == project_root:
        raise RuntimeError("Could not find project root containing the market_helper package.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


def load_local_env_values(root: Path) -> Dict[str, str]:
    candidates: Iterable[Path] = (
        root / "configs" / "portfolio_monitor" / "local.env",
        root / "configs" / "local.env",
    )
    values: Dict[str, str] = {}
    for path in candidates:
        if not path.exists():
            continue
        for raw_line in path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            if line.startswith("export "):
                line = line[len("export "):].strip()
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip().strip(chr(34)).strip("'")
        if values:
            break
    return values


local_env_values = load_local_env_values(project_root)
lookthrough_path = project_root / "configs" / "portfolio_monitor" / "us_sector_lookthrough.json"

TARGET_SYMBOL = "QQQ"
RUN_LIVE_FETCH = True
WRITE_LOOKTHROUGH_STORE = False

ALPHA_VANTAGE_API_KEY = (
    os.environ.get("ALPHA_VANTAGE_API_KEY", "").strip()
    or local_env_values.get("ALPHA_VANTAGE_API_KEY", "").strip()
)

print(f"Project root: {project_root}")
print(f"Lookthrough store: {lookthrough_path}")
print(f"Target symbol: {TARGET_SYMBOL}")
print(f"RUN_LIVE_FETCH: {RUN_LIVE_FETCH}")
print(f"WRITE_LOOKTHROUGH_STORE: {WRITE_LOOKTHROUGH_STORE}")
print(f"ALPHA_VANTAGE_API_KEY detected: {'yes' if ALPHA_VANTAGE_API_KEY else 'no'}")


Project root: /Users/kelvin/git_projects/market_helper
Lookthrough store: /Users/kelvin/git_projects/market_helper/configs/portfolio_monitor/us_sector_lookthrough.json
Target symbol: QQQ
RUN_LIVE_FETCH: True
WRITE_LOOKTHROUGH_STORE: False
ALPHA_VANTAGE_API_KEY detected: yes


In [2]:
import json

from IPython.display import display
import pandas as pd


def load_cached_symbol_entry(path: Path, symbol: str) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    symbols = payload.get("symbols", {}) if isinstance(payload, dict) else {}
    if not isinstance(symbols, dict):
        return {}
    return dict(symbols.get(symbol.upper(), {}))


cached_entry = load_cached_symbol_entry(lookthrough_path, TARGET_SYMBOL)

if not cached_entry:
    print(f"No cached lookthrough entry found for {TARGET_SYMBOL}.")
else:
    summary_frame = pd.DataFrame(
        [
            {
                "symbol": TARGET_SYMBOL,
                "status": cached_entry.get("status", ""),
                "updated_at": cached_entry.get("updated_at", ""),
                "last_attempted_at": cached_entry.get("last_attempted_at", ""),
                "error_message": cached_entry.get("error_message", ""),
            }
        ]
    )
    sector_frame = pd.DataFrame(cached_entry.get("sectors", []))
    print("Cached lookthrough summary")
    display(summary_frame)
    if sector_frame.empty:
        print("Cached sector weights are empty.")
    else:
        display(sector_frame)


Cached lookthrough summary


,symbol,status,updated_at,last_attempted_at,error_message
0,QQQ,ok,2026-04-09,2026-04-09,


,sector,weight
0,Technology,0.486
1,Communication Services,0.159
2,Consumer Discretionary,0.117
3,Consumer Staples,0.083
4,Health Care,0.051
5,Industrials,0.037
6,Utilities,0.015
7,Materials,0.013
8,Energy,0.007
9,Financials,0.002


In [3]:
from market_helper.data_sources.alpha_vantage import AlphaVantageClient
from market_helper.domain.portfolio_monitor.services.etf_sector_lookthrough import (
    ALPHA_VANTAGE_SECTOR_TO_INTERNAL_BUCKET,
    sync_us_sector_lookthrough_from_alpha_vantage,
)


def normalize_sector_name(raw_sector: str) -> str:
    cleaned = str(raw_sector).strip()
    return ALPHA_VANTAGE_SECTOR_TO_INTERNAL_BUCKET.get(cleaned.lower(), cleaned)


live_frame = pd.DataFrame()
live_error = None

if RUN_LIVE_FETCH:
    if not ALPHA_VANTAGE_API_KEY:
        print("ALPHA_VANTAGE_API_KEY is missing, so the notebook will use only the cached entry shown above.")
    else:
        try:
            client = AlphaVantageClient(api_key=ALPHA_VANTAGE_API_KEY)
            rows = client.fetch_etf_sector_weightings(TARGET_SYMBOL)
            live_frame = pd.DataFrame(
                [
                    {
                        "symbol": row.symbol,
                        "alpha_vantage_sector": row.sector,
                        "sector": normalize_sector_name(row.sector),
                        "weight": float(row.weight),
                    }
                    for row in rows
                ]
            )
            if live_frame.empty:
                print(f"Alpha Vantage returned no rows for {TARGET_SYMBOL}.")
            else:
                normalized_frame = (
                    live_frame.groupby(["symbol", "sector"], as_index=False)["weight"]
                    .sum()
                    .sort_values(["weight", "sector"], ascending=[False, True], ignore_index=True)
                )
                print("Live Alpha Vantage sector weights")
                display(normalized_frame)
        except Exception as exc:
            live_error = exc
            print("Live fetch failed cleanly; the cached lookthrough preview above is still usable.")
            print(type(exc).__name__, exc)
else:
    print("RUN_LIVE_FETCH is False. Set it to True to call Alpha Vantage from the notebook.")

if WRITE_LOOKTHROUGH_STORE:
    if not ALPHA_VANTAGE_API_KEY:
        print("WRITE_LOOKTHROUGH_STORE was requested, but ALPHA_VANTAGE_API_KEY is missing.")
    elif live_error is not None:
        print("WRITE_LOOKTHROUGH_STORE was skipped because the live fetch failed.")
    else:
        sync_us_sector_lookthrough_from_alpha_vantage(
            symbols=[TARGET_SYMBOL],
            output_path=lookthrough_path,
            api_key=ALPHA_VANTAGE_API_KEY,
            force_refresh=True,
        )
        refreshed_entry = load_cached_symbol_entry(lookthrough_path, TARGET_SYMBOL)
        print(f"Wrote refreshed lookthrough store entry for {TARGET_SYMBOL}.")
        display(pd.DataFrame(refreshed_entry.get("sectors", [])))
else:
    print("WRITE_LOOKTHROUGH_STORE is False, so the repo JSON file was not modified by this notebook run.")


Live Alpha Vantage sector weights


,symbol,sector,weight
0,QQQ,Technology,0.486
1,QQQ,Communication Services,0.159
2,QQQ,Consumer Discretionary,0.117
3,QQQ,Consumer Staples,0.083
4,QQQ,Health Care,0.051
5,QQQ,Industrials,0.037
6,QQQ,Utilities,0.015
7,QQQ,Materials,0.013
8,QQQ,Energy,0.007
9,QQQ,Financials,0.002


WRITE_LOOKTHROUGH_STORE is False, so the repo JSON file was not modified by this notebook run.
